## Étape 3 — Découpage

In [1]:
import json

with open("../data/pages_propres.json", encoding="utf-8") as f:
    pages = json.load(f)
print(len(pages), "pages chargées")

23 pages chargées


## Code : Q5, la fonction decouper

In [10]:
def decouper(texte: str, taille: int, chevauchement: int) -> list[str]:
    """
    Coupe texte en morceaux de `taille` caractères qui se chevauchent de `chevauchement` caractères.
    """
    if chevauchement >= taille:
        raise ValueError("Le chevauchement doit être inférieur à la taille")
    if not texte:
        return []

    pas = taille - chevauchement
    chunks = []
    debut = 0
    while True:
        chunks.append(texte[debut:debut + taille])
        if debut + taille >= len(texte):   
            break
        debut += pas
    return chunks

In [11]:
resultat = decouper("abcdefghij", taille=4, chevauchement=1)
print(resultat)
assert resultat == ["abcd", "defg", "ghij"]

['abcd', 'defg', 'ghij']


## Code : Q6, découper tout le corpus avec trois tailles

In [12]:
def construire_chunks(pages: list[dict], taille: int, chevauchement: int) -> list[dict]:
    """Découpe chaque page séparément, pour garder source / page / service sur chaque chunk."""
    chunks = []
    for p in pages:
        for i, morceau in enumerate(decouper(p["texte"], taille, chevauchement)):
            chunks.append({
                "id": f"{p['source']}-p{p['page']}-{i}",   
                "texte": morceau,
                "source": p["source"],
                "page": p["page"],
                "service": p["service"],
            })
    return chunks

print(f"{'Taille':>7} | {'Chevauch.':>9} | {'Nb chunks':>9} | {'Longueur moy.':>13}")
print("-" * 48)
for taille, chev in [(200, 20), (500, 50), (1000, 100)]:
    chunks = construire_chunks(pages, taille, chev)
    moyenne = sum(len(c["texte"]) for c in chunks) / len(chunks)
    print(f"{taille:>7} | {chev:>9} | {len(chunks):>9} | {moyenne:>13.0f}")

 Taille | Chevauch. | Nb chunks | Longueur moy.
------------------------------------------------
    200 |        20 |       156 |           189
    500 |        50 |        69 |           423
   1000 |       100 |        38 |           746


## Code : Q7, choix de la taille et sauvegarde:

Chunk trop petit : il perd son contexte, la réponse est coupée en deux, et il y en a beaucoup, donc plus de bruit.
Chunk trop grand : il mélange plusieurs sujets, son vecteur ne représente aucun sujet précis, et on envoie beaucoup de texte inutile au LLM.
Chevauchement : il évite de perdre une phrase coupée à la frontière entre deux chunks, car elle apparaît entière dans l’un des deux.
Taille retenue : 500 caractères avec 50 de chevauchement.

In [16]:
TAILLE, CHEVAUCHEMENT = 500, 50     

chunks = construire_chunks(pages, TAILLE, CHEVAUCHEMENT)
print(len(chunks), "chunks")

assert len({c["id"] for c in chunks}) == len(chunks)

for c in chunks[:2]:
    print("-" * 60)
    print(c["id"], "|", c["service"])
    print(c["texte"])

with open("../data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print("\nSauvegardé dans ../data/chunks.json")

69 chunks
------------------------------------------------------------
horaires.txt-p1-0 | General
HORAIRES D'OUVERTURE DES SITES
Altéa Services — Services généraux — Référence GEN-004 — mise à jour du 1er septembre 2026

1. Horaires des sites
Site de Paris (siège) : ouvert du lundi au vendredi de 7 h à 20 h. Accueil des visiteurs de 8 h 30 à 18 h.
Site de Lyon : ouvert du lundi au vendredi de 7 h 30 à 19 h. Accueil des visiteurs de 9 h à 17 h 30.
Site de Nantes : ouvert du lundi au vendredi de 8 h à 19 h. Accueil des visiteurs de 9 h à 17 h.
Les sites sont fermés le week-end et les jours fér
------------------------------------------------------------
horaires.txt-p1-1 | General
Les sites sont fermés le week-end et les jours fériés. Un accès exceptionnel le samedi peut être demandé aux services généraux au moins 3 jours à l'avance.

2. Horaires de travail
La durée de référence est de 35 heures par semaine pour les salariés non cadres. Les plages fixes, pendant lesquelles chacun est pr